# Preparação dos dados

Execute este notebook em ordem para refazer **limpeza → FinBERT → integração**. Os arquivos de origem e as probabilidades já calculadas estão incluídos; o artigo pode ser executado diretamente com as bases preparadas.

Cada etapa explica suas transformações e confere chaves e valores. Os dados originais e a revisão humana são preservados. Fontes e identificadores: [proveniência](../docs/proveniencia.json).

## 1. Preparar notícias e cotações

Usamos NVDA, NFLX, BAC, JNJ e KO de 2017 a 2019. A notícia é uma associação
empresa/data/manchete; a cotação é uma observação empresa/pregão. Os arquivos
de origem ficam em `data/raw/` e são apenas lidos.

O recorte incluído conserva os campos originais e a posição de cada linha no
CSV completo do [Kaggle](https://www.kaggle.com/datasets/miguelaenlle/massive-stock-news-analysis-db-for-nlpbacktests).
Ele ainda contém as quatro duplicatas: filtrar empresas e anos não substitui
a limpeza abaixo. Se o CSV completo estiver disponível, o mesmo código também o aceita.


### 1.1 Configuração

Os caminhos funcionam ao abrir o notebook na raiz ou na pasta `notebooks/`.
Definimos o período uma vez e criamos somente a pasta de resultados.


In [1]:
import os
os.environ['MKL_THREADING_LAYER'] = 'SEQUENTIAL'
import json
import hashlib
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display

RAIZ = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
ORIGINAIS = RAIZ / 'data' / 'raw'
TRATADOS = RAIZ / 'data' / 'processed'
TRATADOS.mkdir(parents=True, exist_ok=True)
TICKERS = ['NVDA', 'NFLX', 'BAC', 'JNJ', 'KO']
INICIO, FIM = pd.Timestamp('2017-01-01'), pd.Timestamp('2019-12-31')

### 1.2 Ler e selecionar o recorte

`source_id` identifica o registro original; `source_row` informa sua posição
no CSV completo, começando em 1 após o cabeçalho. Ao usar o recorte fornecido,
preservamos essa posição, sem renumerá-la. Normalizamos os tickers e validamos
as datas. Usamos o dia escrito na fonte; a validação em UTC não desloca o dia
analítico, pois não dispomos de horário confiável de publicação.


In [2]:
entrada = ORIGINAIS / 'noticias_recorte_original.csv'
if not entrada.exists():
    entrada = ORIGINAIS / 'raw_analyst_ratings.csv'
noticias = pd.read_csv(entrada, dtype='string', keep_default_na=False)
obrigatorias = {'Unnamed: 0', 'headline', 'url', 'publisher', 'date', 'stock'}
assert obrigatorias.issubset(noticias.columns), 'Faltam colunas no CSV de notícias.'
noticias = noticias.rename(columns={'Unnamed: 0': 'source_id'})
if 'source_row' not in noticias:
    noticias['source_row'] = np.arange(1, len(noticias) + 1)
for coluna in ['source_id', 'source_row']:
    numeros = pd.to_numeric(noticias[coluna], errors='raise')
    assert numeros.mod(1).eq(0).all(), f'{coluna} deve conter inteiros.'
    noticias[coluna] = numeros.astype('int64')
assert noticias['source_id'].is_unique, 'Há identificadores de origem repetidos.'
noticias['stock_original'] = noticias['stock']
noticias['stock'] = noticias['stock'].str.strip().str.upper()
noticias = noticias.loc[noticias['stock'].isin(TICKERS)].copy()
noticias['date_original'] = noticias['date']
texto_data = noticias['date_original'].str.strip()
valida = pd.to_datetime(texto_data, format='mixed', errors='coerce', utc=True).notna()
dia = texto_data.str.extract(r'^(\d{4}-\d{2}-\d{2})(?:$|[ T])', expand=False)
noticias['news_date'] = pd.to_datetime(dia, format='%Y-%m-%d', errors='coerce')

### 1.3 Limpar textos e registrar exclusões

Preservamos a manchete original e ajustamos apenas espaços. Pontuação,
números e negações são mantidos para o FinBERT. URL ou fonte ausente recebe
um sinalizador e não causa exclusão. Cada exclusão tem um motivo; na base de
referência, não há data inválida nem manchete vazia.


In [3]:
noticias['headline_original'] = noticias['headline']
noticias['headline'] = noticias['headline'].str.replace(r'\s+', ' ', regex=True).str.strip()
noticias['url_ausente'] = noticias['url'].str.strip().eq('')
noticias['publisher_ausente'] = noticias['publisher'].str.strip().eq('')
motivo = pd.Series(np.select([
    ~valida | noticias['news_date'].isna(),
    ~noticias['news_date'].between(INICIO, FIM),
    noticias['headline'].eq('')
], ['data_invalida', 'fora_do_periodo', 'manchete_vazia'], default=''), index=noticias.index)
excluidas = noticias.loc[motivo.ne('')].copy()
excluidas['motivo'] = motivo.loc[excluidas.index]
excluidas['source_id_mantido'] = pd.NA
noticias = noticias.loc[motivo.eq('')].copy()
antes_duplicatas = len(noticias)

### 1.4 Remover duplicatas e conservar associações entre empresas

A chave é empresa/data/manchete. Mantemos o menor `source_id` e registramos
a ligação entre a cópia retirada e a mantida. A mesma manchete em empresas
diferentes permanece: é uma associação compartilhada, não uma duplicata
dentro da empresa. A referência passa de 6.556 para 6.552 associações.


In [4]:
CHAVE = ['stock', 'news_date', 'headline']
noticias = noticias.sort_values('source_id').copy()
id_mantido = noticias.groupby(CHAVE)['source_id'].transform('min')
duplicada = noticias.duplicated(CHAVE, keep='first')
duplicatas = noticias.loc[duplicada].copy()
duplicatas['motivo'] = 'duplicata_empresa_data_manchete'
duplicatas['source_id_mantido'] = id_mantido.loc[duplicada]
auditoria = pd.concat([excluidas, duplicatas], ignore_index=True)
auditoria['source_id_mantido'] = pd.to_numeric(auditoria['source_id_mantido']).astype('Int64')
noticias = noticias.loc[~duplicada].copy()
noticias['n_empresas_manchete'] = noticias.groupby(['news_date', 'headline'])['stock'].transform('nunique')
noticias['compartilhada_entre_empresas'] = noticias['n_empresas_manchete'].gt(1)
noticias = noticias.drop(columns='date').sort_values(['stock', 'news_date', 'source_id']).reset_index(drop=True)
assert noticias['source_id'].is_unique and not noticias.duplicated(CHAVE).any()
noticias.to_csv(TRATADOS / 'noticias_limpas.csv', index=False, date_format='%Y-%m-%d')
auditoria.to_csv(TRATADOS / 'exclusoes_noticias.csv', index=False, date_format='%Y-%m-%d')
print(f'No recorte: {antes_duplicatas}; duplicatas retiradas: {len(duplicatas)}; mantidas: {len(noticias)}.')
display(noticias.groupby('stock').size().reindex(TICKERS).rename('noticias').to_frame())

No recorte: 6556; duplicatas retiradas: 4; mantidas: 6552.


,noticias
stock,
NVDA,1647
NFLX,2344
BAC,871
JNJ,1046
KO,644


### 1.5 Ler as cotações preservadas

Reutilizamos o snapshot do Yahoo Finance, obtido via `yfinance`, com seus
metadados e hash. Não refazemos a coleta: preços ajustados podem mudar na
fonte. O intervalo inclui dias anteriores e posteriores a 2017–2019 para
calcular retornos nas bordas. `Adj Close` é o fechamento ajustado fornecido
pela fonte; não aplicamos um ajuste próprio.


In [5]:
arquivo_precos = ORIGINAIS / 'cotacoes_yahoo_margem.csv'
coleta_precos = json.loads((ORIGINAIS / 'cotacoes_yahoo_margem.json').read_text(encoding='utf-8'))
assert coleta_precos['tickers'] == TICKERS, 'O snapshot contém outro conjunto de empresas.'
assert hashlib.sha256(arquivo_precos.read_bytes()).hexdigest() == coleta_precos['sha256'], 'O snapshot foi alterado.'
precos_brutos = pd.read_csv(arquivo_precos, keep_default_na=False)
precos = precos_brutos.rename(columns={'Adj Close': 'adj_close', 'Volume': 'volume'}).copy()
precos['price_source_row'] = np.arange(1, len(precos) + 1)
precos['trade_date_original'] = precos['trade_date'].astype('string')
precos['trade_date'] = pd.to_datetime(precos['trade_date_original'].str[:10], format='%Y-%m-%d', errors='raise')
precos['stock'] = precos['stock'].astype('string').str.strip().str.upper()
precos[['adj_close', 'volume']] = precos[['adj_close', 'volume']].apply(pd.to_numeric, errors='raise')
precos['volume_invalido'] = ~np.isfinite(precos['volume']) | precos['volume'].lt(0) | precos['volume'].mod(1).ne(0)
precos['volume_zero'] = precos['volume'].eq(0)

### 1.6 Validar e salvar as cotações

Exigimos preços positivos, volumes válidos e margens temporais. Duplicatas
idênticas podem ser retiradas; valores divergentes interrompem a execução.
Não preenchemos preços ausentes. A referência tem 754 pregões por empresa no
período. A conferência exata das datas com o calendário XNYS ocorre na
integração, adiante; só contar linhas não detectaria toda lacuna.


In [6]:
assert precos['stock'].isin(TICKERS).all(), 'Há empresas fora do recorte nas cotações.'
assert precos['trade_date'].between('2016-12-15', '2020-01-10').all(), 'Há datas fora da margem.'
assert (np.isfinite(precos['adj_close']) & precos['adj_close'].gt(0)).all(), 'Há preços inválidos.'
assert not precos['volume_invalido'].any(), 'Há volumes inválidos.'
chave_preco = ['stock', 'trade_date']
repetidos = precos.loc[precos.duplicated(chave_preco, keep=False)]
campos_valor = [c for c in precos_brutos.columns if c not in chave_preco]
campos_valor = [{'Adj Close': 'adj_close', 'Volume': 'volume'}.get(c, c) for c in campos_valor]
assert not repetidos.groupby(chave_preco)[campos_valor].nunique(dropna=False).gt(1).any().any(), 'Há cotações divergentes.'
precos = precos.drop_duplicates(chave_preco).sort_values(chave_preco).reset_index(drop=True)
precos['no_recorte'] = precos['trade_date'].between(INICIO, FIM)
cobertura_precos = precos.loc[precos['no_recorte']].groupby('stock').size().reindex(TICKERS)
assert cobertura_precos.eq(754).all(), 'Confira a cobertura de 2017–2019.'
for ticker in TICKERS:
    dias = precos.loc[precos['stock'].eq(ticker), 'trade_date']
    assert dias.lt(INICIO).any() and dias.gt(FIM).any(), f'Falta margem temporal em {ticker}.'
precos.to_csv(TRATADOS / 'cotacoes_limpas_com_margem.csv', index=False, date_format='%Y-%m-%d')
display(cobertura_precos.rename('pregoes_2017_2019').to_frame())

,pregoes_2017_2019
stock,
NVDA,754
NFLX,754
BAC,754
JNJ,754
KO,754


## 2. Sentimento das manchetes com FinBERT

Aplicamos o modelo pronto `ProsusAI/finbert`, sem treinamento adicional, à manchete inteira. A revisão dos pesos e do tokenizador é fixada. Cada texto distinto é classificado uma vez, mas todas as associações entre manchetes e empresas são preservadas. As probabilidades softmax não são probabilidades calibradas de acerto.

O cache em `data/raw/` conserva somente texto, hash e três probabilidades. Sua configuração identifica modelo, revisão, tokenização, precisão numérica e versões de inferência; resultados incompatíveis não são reutilizados. O tokenizador e, quando necessário, os pesos são baixados na primeira utilização. `SOMENTE_CACHE=True` exige esses arquivos já disponíveis no computador.

In [7]:
from importlib.metadata import version
from transformers import AutoTokenizer

MODELO = 'ProsusAI/finbert'
REVISAO = '4556d13015211d73dccd3fdd39d39232506f3e43'
SOMENTE_CACHE = False
MAX_TOKENS, TAMANHO_LOTE, TOL_EMPATE = 512, 16, 1e-8
PROB_COLS = ['p_positivo', 'p_negativo', 'p_neutro']
CLASSES = np.array(['positivo', 'negativo', 'neutro'])
noticias = pd.read_csv(TRATADOS / 'noticias_limpas.csv', parse_dates=['news_date'], keep_default_na=False)
COLUNAS_ORIGINAIS = noticias.columns.tolist()
assert noticias['source_id'].is_unique and noticias['news_date'].notna().all()
assert noticias['headline'].str.strip().ne('').all()
assert not noticias.duplicated(['stock', 'news_date', 'headline']).any()
noticias['headline_hash'] = noticias['headline'].map(lambda s: hashlib.sha256(s.encode('utf-8')).hexdigest())
unicas = noticias[['headline_hash', 'headline']].drop_duplicates().copy()
assert unicas['headline_hash'].is_unique
tokenizador = AutoTokenizer.from_pretrained(MODELO, revision=REVISAO, local_files_only=SOMENTE_CACHE)
unicas['n_tokens'] = tokenizador(unicas['headline'].tolist(), truncation=False,
                               padding=False, return_length=True)['length']
unicas['foi_truncada'] = unicas['n_tokens'].gt(MAX_TOKENS)
print(f'{len(noticias):,} associações; {len(unicas):,} textos; {unicas["foi_truncada"].sum()} truncados.')

6,552 associações; 6,019 textos; 0 truncados.


### Cache e classificação

Conferimos a configuração, a correspondência entre texto e hash e a validade das probabilidades antes de usar o cache. Textos ausentes são processados em CPU, com precisão `float32`, lotes de 16 e corte em 512 tokens. Consultamos os rótulos do próprio modelo para identificar as classes. O progresso é gravado a cada 16 lotes; um arquivo temporário evita substituir o cache por uma gravação incompleta.

In [8]:
def validar_probabilidades(tabela):
    p = tabela[PROB_COLS].to_numpy(dtype=float)
    assert np.isfinite(p).all() and ((p >= 0) & (p <= 1)).all()
    assert np.allclose(p.sum(axis=1), 1, atol=1e-5, rtol=0)

config_cache = {
    'modelo': MODELO, 'revisao_modelo': REVISAO, 'revisao_tokenizador': REVISAO,
    'max_tokens': MAX_TOKENS, 'truncation': True, 'dtype': 'float32',
    **{p: version(p) for p in ['transformers', 'torch', 'tokenizers']}}
ARQ_CACHE = RAIZ / 'data/raw/finbert_probabilidades.csv'
ARQ_CONFIG = RAIZ / 'data/raw/finbert_configuracao.json'
COL_CACHE = ['headline_hash', 'headline', *PROB_COLS]
cache = pd.DataFrame(columns=COL_CACHE)
if ARQ_CACHE.exists():
    assert ARQ_CONFIG.is_file(), 'Cache sem configuração: restaure o par de arquivos ou remova o cache.'
    assert json.loads(ARQ_CONFIG.read_text(encoding='utf-8')) == config_cache, 'Cache incompatível com a configuração atual.'
    cache = pd.read_csv(ARQ_CACHE, keep_default_na=False)
    assert cache['headline_hash'].is_unique
    assert cache['headline_hash'].equals(cache['headline'].map(lambda s: hashlib.sha256(s.encode('utf-8')).hexdigest()))
    validar_probabilidades(cache)
cache = cache.loc[cache['headline_hash'].isin(unicas['headline_hash']), COL_CACHE].copy()
pendentes = unicas.loc[~unicas['headline_hash'].isin(cache['headline_hash'])].sort_values(['n_tokens', 'headline_hash'])
registros = cache.to_dict('records')

def salvar_cache():
    tabela = pd.DataFrame(registros, columns=COL_CACHE).sort_values('headline_hash').reset_index(drop=True)
    assert tabela['headline_hash'].is_unique
    validar_probabilidades(tabela)
    ARQ_CACHE.parent.mkdir(parents=True, exist_ok=True)
    ARQ_CONFIG.write_text(json.dumps(config_cache, indent=2, ensure_ascii=False), encoding='utf-8')
    temporario = ARQ_CACHE.with_suffix('.tmp')
    tabela.to_csv(temporario, index=False, float_format='%.10g')
    temporario.replace(ARQ_CACHE)

print(f'{len(cache):,} textos reutilizados; {len(pendentes):,} a classificar.')
if len(pendentes):
    import torch
    from transformers import AutoModelForSequenceClassification
    torch.set_num_threads(min(4, os.cpu_count() or 1))
    modelo = AutoModelForSequenceClassification.from_pretrained(
        MODELO, revision=REVISAO, local_files_only=SOMENTE_CACHE, use_safetensors=False
    ).to(device='cpu', dtype=torch.float32).eval()
    indices = {str(nome).lower(): int(i) for i, nome in modelo.config.id2label.items()}
    assert set(indices) == {'positive', 'negative', 'neutral'}
    for numero, inicio in enumerate(range(0, len(pendentes), TAMANHO_LOTE), start=1):
        lote = pendentes.iloc[inicio:inicio + TAMANHO_LOTE]
        entradas = tokenizador(lote['headline'].tolist(), padding=True, truncation=True,
                               max_length=MAX_TOKENS, return_tensors='pt')
        with torch.inference_mode():
            p = torch.softmax(modelo(**entradas).logits.float(), dim=-1).cpu().numpy()
        resultado = lote[['headline_hash', 'headline']].copy()
        for coluna, classe in zip(PROB_COLS, ['positive', 'negative', 'neutral']):
            resultado[coluna] = p[:, indices[classe]]
        registros.extend(resultado[COL_CACHE].to_dict('records'))
        if numero % 16 == 0 or inicio + TAMANHO_LOTE >= len(pendentes):
            salvar_cache()
            print(f'{len(registros):,}/{len(unicas):,} textos salvos.', flush=True)
if not ARQ_CACHE.exists():
    salvar_cache()
probabilidades = pd.read_csv(ARQ_CACHE, keep_default_na=False)
probabilidades = probabilidades.loc[probabilidades['headline_hash'].isin(unicas['headline_hash'])].copy()
assert probabilidades['headline_hash'].is_unique
assert set(probabilidades['headline_hash']) == set(unicas['headline_hash'])
validar_probabilidades(probabilidades)

6,019 textos reutilizados; 0 a classificar.


### Medidas e associação às notícias

O escore é `P(positivo) − P(negativo)`. A classe corresponde à maior probabilidade, com empate quando há máximos à tolerância de `1e-8`. Guardamos também a margem entre as duas maiores probabilidades e os indicadores de truncamento. Um escore próximo de zero pode resultar de tons opostos e não define neutralidade.

Devolvemos as probabilidades às associações originais sem alterar textos, IDs, empresas ou datas. O agregado por empresa/data civil usa médias das probabilidades e contagens; ausência de notícias não recebe classe neutra. A integração a seguir reagrupará as manchetes por pregão.

In [9]:
def acrescentar_medidas(tabela, coluna_classe):
    tabela = tabela.copy()
    validar_probabilidades(tabela)
    p = tabela[PROB_COLS].to_numpy(dtype=float)
    maiores = p.max(axis=1)
    empates = np.isclose(p, maiores[:, None], atol=TOL_EMPATE, rtol=0).sum(axis=1) > 1
    tabela[coluna_classe] = np.where(empates, 'empate', CLASSES[p.argmax(axis=1)])
    tabela['escore_sentimento'] = tabela['p_positivo'] - tabela['p_negativo']
    tabela['probabilidade_maxima'] = maiores
    tabela['margem_top2'] = np.diff(np.sort(p, axis=1)[:, -2:], axis=1).ravel()
    tabela['empate_probabilidades'] = empates
    return tabela

medidas = acrescentar_medidas(probabilidades, 'classe_finbert').drop(columns='headline')
medidas = medidas.merge(unicas[['headline_hash', 'n_tokens', 'foi_truncada']],
                        on='headline_hash', validate='one_to_one')
classificadas = noticias.merge(medidas, on='headline_hash', how='left', validate='many_to_one', sort=False)
pd.testing.assert_frame_equal(classificadas[COLUNAS_ORIGINAIS], noticias[COLUNAS_ORIGINAIS])
assert classificadas['source_id'].is_unique
validar_probabilidades(classificadas)
classificadas['modelo_finbert'], classificadas['revisao_finbert'] = MODELO, REVISAO
classificadas.to_csv(TRATADOS / 'noticias_com_sentimento.csv', index=False,
                     date_format='%Y-%m-%d', float_format='%.10g')
diario = classificadas.groupby(['stock', 'news_date'], as_index=False).agg(
    p_positivo=('p_positivo', 'mean'), p_negativo=('p_negativo', 'mean'), p_neutro=('p_neutro', 'mean'),
    n_noticias=('source_id', 'size'), n_compartilhadas=('compartilhada_entre_empresas', 'sum'),
    n_truncadas=('foi_truncada', 'sum'))
diario = acrescentar_medidas(diario, 'classe_diaria')
assert diario['n_noticias'].sum() == len(classificadas)
diario.to_csv(TRATADOS / 'sentimento_diario_noticias.csv', index=False,
              date_format='%Y-%m-%d', float_format='%.10g')
display(classificadas['classe_finbert'].value_counts().rename('associacoes').to_frame())

,associacoes
classe_finbert,
neutro,3598
positivo,1613
negativo,1341


### Revisão humana já realizada

A revisão dirigida de 50 manchetes está preservada no arquivo original em `docs/revisao_manual/`; não sorteamos outra amostra nem sobrescrevemos as anotações. Conferimos sua correspondência com os textos classificados e resumimos a concordância. Um único revisor, a seleção dirigida e a exposição às previsões impedem interpretar essa contagem como acurácia geral. Os rótulos humanos não substituem as classes automáticas.

O FinBERT classifica o texto inteiro. Uma manchete associada a várias empresas pode ter tom diferente para cada uma. Consulte o [relatório FinBERT](../docs/finbert.md), o [modelo](https://huggingface.co/ProsusAI/finbert/tree/4556d13015211d73dccd3fdd39d39232506f3e43) e o [artigo original](https://arxiv.org/abs/1908.10063).

In [10]:
ARQ_REVISAO = RAIZ / 'docs/revisao_manual/amostra_50_cf0a9799_e1f584a3.csv'
revisao = pd.read_csv(ARQ_REVISAO, sep=None, engine='python', encoding='utf-8-sig',
                      keep_default_na=False, dtype=str)
assert len(revisao) == 50 and revisao['source_id'].is_unique
origem = classificadas.assign(source_id=classificadas['source_id'].astype(str)).set_index('source_id')
assert revisao['source_id'].isin(origem.index).all()
for coluna in ['headline', 'stock', 'classe_finbert']:
    assert revisao[coluna].eq(revisao['source_id'].map(origem[coluna])).all()
humanos = revisao['rotulo_humano'].str.strip().str.lower()
assert humanos.isin(['positivo', 'negativo', 'neutro', 'ambiguo', '']).all()
comparaveis = humanos.isin(['positivo', 'negativo', 'neutro'])
concordantes = humanos.eq(revisao['classe_finbert']) & comparaveis
print(f'Revisão preservada: {comparaveis.sum()} comparáveis, {concordantes.sum()} concordâncias, '
      f'{(comparaveis & ~concordantes).sum()} divergências. Amostra dirigida; não estima acurácia geral.')

Revisão preservada: 50 comparáveis, 39 concordâncias, 11 divergências. Amostra dirigida; não estima acurácia geral.


## 3. Integrar notícias, pregões e retornos

Esta etapa usa os CSVs preparados acima. Cada manchete é associada ao primeiro pregão **estritamente posterior** à sua data; depois reunimos as manchetes de cada empresa/pregão. Datas sem horário permitem uma comparação exploratória, sem identificar reação causal.


In [11]:
import exchange_calendars as xcals

TICKERS = ['BAC', 'JNJ', 'KO', 'NFLX', 'NVDA']
noticias = pd.read_csv(TRATADOS / 'noticias_com_sentimento.csv', parse_dates=['news_date'], keep_default_na=False)
precos = pd.read_csv(TRATADOS / 'cotacoes_limpas_com_margem.csv', parse_dates=['trade_date'])
precos = precos.sort_values(['stock', 'trade_date']).reset_index(drop=True)
assert set(noticias.stock) == set(precos.stock) == set(TICKERS)
assert noticias.source_id.is_unique and not noticias.duplicated(['stock', 'news_date', 'headline']).any()
assert not precos.duplicated(['stock', 'trade_date']).any()
assert noticias.news_date.between(INICIO, FIM).all() and noticias.news_date.eq(noticias.news_date.dt.normalize()).all()
assert precos.trade_date.notna().all() and precos.trade_date.eq(precos.trade_date.dt.normalize()).all()
assert np.isfinite(precos.adj_close).all() and precos.adj_close.gt(0).all()
assert noticias[PROB_COLS].ge(0).all().all() and noticias[PROB_COLS].le(1).all().all()
assert np.allclose(noticias[PROB_COLS].sum(axis=1), 1, atol=1e-6)
assert np.allclose(noticias.escore_sentimento, noticias.p_positivo - noticias.p_negativo, atol=1e-8)

### Calendário e retornos

Conferimos as cotações contra o calendário XNYS: finais de semana e feriados não são pregões. Calculamos a variação percentual do fechamento ajustado dentro de cada empresa, **antes** do recorte de 2017–2019, preservando o preço anterior ao primeiro retorno. O módulo do retorno mede a oscilação diária; não interpolamos preços.


In [12]:
calendario = xcals.get_calendar('XNYS', start=precos.trade_date.min(), end=precos.trade_date.max())
sessoes_esperadas = pd.DatetimeIndex(calendario.sessions).tz_localize(None).normalize()
for ticker in TICKERS:
    observadas = pd.DatetimeIndex(precos.loc[precos.stock.eq(ticker), 'trade_date'])
    assert observadas.equals(sessoes_esperadas), f'Cotações incompletas ou datas extras: {ticker}'
precos['trade_date_anterior'] = precos.groupby('stock')['trade_date'].shift(1)
precos['adj_close_anterior'] = precos.groupby('stock')['adj_close'].shift(1)
precos['retorno_pct'] = 100 * (precos.adj_close / precos.adj_close_anterior - 1)
precos['retorno_absoluto_pct'] = precos.retorno_pct.abs()
precos['retorno_no_recorte'] = precos.trade_date.between(INICIO, FIM)
assert np.isfinite(precos.loc[precos.retorno_no_recorte, 'retorno_pct']).all()

### Associar a notícia ao próximo pregão

A busca exige `pregão anterior ≤ data da notícia < pregão de resposta`. Uma notícia de sexta pode apontar para segunda; feriados podem adiar essa data. Mantemos as notícias de 31/12/2019 ligadas a janeiro de 2020, identificadas como margem fora do painel principal.


In [13]:
datas = noticias[['stock', 'news_date']].drop_duplicates().sort_values(['stock', 'news_date']).copy()
datas['trade_date'] = pd.NaT
for ticker in TICKERS:
    mask = datas.stock.eq(ticker)
    sessoes = precos.loc[precos.stock.eq(ticker), 'trade_date'].to_numpy(dtype='datetime64[ns]')
    pos = np.searchsorted(sessoes, datas.loc[mask, 'news_date'].to_numpy(dtype='datetime64[ns]'), side='right')
    assert (pos < len(sessoes)).all(), f'Sem cotação do pregão posterior: {ticker}'
    datas.loc[mask, 'trade_date'] = sessoes[pos]
campos_preco = ['stock', 'trade_date', 'trade_date_anterior', 'adj_close', 'adj_close_anterior',
                'retorno_pct', 'retorno_absoluto_pct', 'retorno_no_recorte']
mapa = noticias.merge(datas, on=['stock', 'news_date'], how='left', validate='many_to_one')
mapa = mapa.merge(precos[campos_preco], on=['stock', 'trade_date'], how='left', validate='many_to_one')
assert len(mapa) == len(noticias) and mapa.source_id.is_unique
assert mapa[['trade_date', 'trade_date_anterior', 'retorno_pct']].notna().all().all()
assert (mapa.trade_date_anterior <= mapa.news_date).all() and (mapa.news_date < mapa.trade_date).all()
mapa['dias_ate_pregao'] = (mapa.trade_date - mapa.news_date).dt.days
mapa['retorno_fora_recorte'] = ~mapa.retorno_no_recorte
mapa = mapa.sort_values(['stock', 'trade_date', 'news_date', 'source_id']).reset_index(drop=True)

### Reunir as manchetes por empresa/pregão

Cada manchete recebe o mesmo peso nas médias das probabilidades. O escore é `P(positivo) − P(negativo)`; a intensidade é seu módulo, e a classe corresponde à maior probabilidade média, com empates explícitos. Não usamos média simples de médias diárias, pois os dias podem ter quantidades diferentes de notícias.


In [14]:
por_pregao = mapa.groupby(['stock', 'trade_date'], as_index=False).agg(
    n_noticias=('source_id', 'size'), n_datas_noticias=('news_date', 'nunique'),
    primeira_news_date=('news_date', 'min'), ultima_news_date=('news_date', 'max'),
    p_positivo_medio=('p_positivo', 'mean'), p_negativo_medio=('p_negativo', 'mean'),
    p_neutro_medio=('p_neutro', 'mean'), n_compartilhadas=('compartilhada_entre_empresas', 'sum'))
por_pregao['escore_sentimento'] = por_pregao.p_positivo_medio - por_pregao.p_negativo_medio
por_pregao['intensidade_sentimento'] = por_pregao.escore_sentimento.abs()
medias = por_pregao[['p_positivo_medio', 'p_negativo_medio', 'p_neutro_medio']].to_numpy()
empates = np.isclose(medias, medias.max(axis=1, keepdims=True), atol=1e-8, rtol=0).sum(axis=1) > 1
classes = np.array(['positivo', 'negativo', 'neutro'])[medias.argmax(axis=1)]
por_pregao['classe_sentimento'] = np.where(empates, 'empate', classes)
por_pregao['empate_probabilidades'] = empates
por_pregao['ao_menos_3_noticias'] = por_pregao.n_noticias.ge(3)
por_pregao = por_pregao.merge(precos[campos_preco], on=['stock', 'trade_date'], how='left', validate='one_to_one')
def marcar_janela(tabela):
    tabela['janela_noticias_completa'] = (tabela.trade_date_anterior >= INICIO) & (tabela.trade_date <= FIM + pd.Timedelta(days=1))
    tabela['elegivel_analise_principal'] = tabela.retorno_no_recorte & tabela.janela_noticias_completa
marcar_janela(por_pregao)
assert por_pregao.n_noticias.sum() == len(noticias)
assert np.allclose(medias.sum(axis=1), 1, atol=1e-6)

### Painel completo e arquivos para o artigo

O painel inclui todos os pregões do recorte. Sem manchetes, somente as contagens recebem zero: sentimento ausente **não** é sentimento neutro. Janelas incompletas nas bordas ficam sinalizadas. As comparações principais usam `tem_noticias & elegivel_analise_principal`; o mínimo de três notícias é contado por empresa/pregão. A tabela por manchete serve à conferência, pois repetir um retorno por notícia não cria observações independentes.


In [15]:
campos_sentimento = [c for c in por_pregao.columns if c not in campos_preco or c in ['stock', 'trade_date']]
painel = precos.loc[precos.retorno_no_recorte].merge(
    por_pregao[campos_sentimento], on=['stock', 'trade_date'], how='left', validate='one_to_one')
painel['tem_noticias'] = painel.n_noticias.notna()
for campo in ['n_noticias', 'n_datas_noticias', 'n_compartilhadas']:
    painel[campo] = painel[campo].fillna(0).astype(int)
for campo in ['empate_probabilidades', 'ao_menos_3_noticias']:
    painel[campo] = painel[campo].fillna(False).astype(bool)
marcar_janela(painel)
painel['situacao_noticias'] = np.where(painel.tem_noticias, 'com_noticias', 'sem_noticias_no_recorte')
sem_noticias = ~painel.tem_noticias
campos_nulos = ['p_positivo_medio', 'p_negativo_medio', 'p_neutro_medio', 'escore_sentimento',
                'intensidade_sentimento', 'classe_sentimento']
assert painel.loc[sem_noticias, campos_nulos].isna().all().all()
assert len(painel) == sessoes_esperadas.to_series().between(INICIO, FIM).sum() * len(TICKERS)
assert not painel.duplicated(['stock', 'trade_date']).any()
arquivos = {'noticias_pregao_retornos.csv': mapa, 'base_integrada_pregoes.csv': painel,
            'sentimento_por_pregao.csv': por_pregao, 'retornos_cotacoes.csv': precos}
for nome, tabela in arquivos.items():
    tabela.to_csv(TRATADOS / nome, index=False, encoding='utf-8', date_format='%Y-%m-%d', float_format='%.12g')
print(f'{len(mapa):,} notícias/empresas; {len(painel):,} pares empresa/pregão; '
      f'{(painel.tem_noticias & painel.elegivel_analise_principal).sum():,} pares elegíveis com notícias.')

6,552 notícias/empresas; 3,770 pares empresa/pregão; 2,387 pares elegíveis com notícias.


As quatro bases foram salvas em `data/processed/`. O artigo pode agora ser executado para gerar as mesmas visualizações e tabelas a partir dessas entradas. Alterações nos dados de origem exigem conferir novamente os resultados e a narrativa.
